In [1]:
from astar_extended import *
import math

In [2]:
class Node():
    count = 0  # keep count of nodes (for use as default names)
    empty = None  # Node(0) returns a default/empty node without connections
    _all = {}  # track all Node objects
    
    def __init__(self, name=None, value=math.inf):
        if name==0 and Node.empty:
            self = Node.empty
        else:
            Node.count += 1
            self.name = name if name is not None else self.count
            self.children, self.parents, self.neighbors = set(), set(), set()
            if name==0:
                Node.empty = self
            else:
                Node._all.update({self.name:self})
    
    def add_neighbor(self, neighbor):
        self.neighbors.add(neighbor)
        neighbor.neighbors.add(self)
    
    def add_neighbors(self, *neighbors):
        for n in neighbors:
            self.add_neighbor(n)
    
    def add_parent(self, parent):
        self.parents.add(parent)
        parent.children.add(self)
    
    def add_parents(self, *parents):
        for p in parents:
            self.add_parent(p)
            
    def add_child(self, child):
        self.children.add(child)
        child.parents.add(self)
    
    def add_children(self, *children):
        for c in children:
            self.add_child(c)
    
    def _print(self):
        return "{} = {}".format(type(self).__name__, self.__dict__)
    
    def __repr__(self):
        return "{} \'{}\'".format(type(self).__name__, self.name)

In [3]:
# Example/test case from https://youtu.be/DhtSZhakyOo by Pieter Abbeel
# Graph provided in video
S, A, B, C, G = Node('s'), Node('a'), Node('b'), Node('c'), Node('g')
S.add_neighbors(B,A)
A.add_neighbors(B,C,G)
B.add_neighbors(A,C)
C.add_neighbors(G)

# Same graph, linked to same goal node
# Node B in this case cannot flow to C, only to A
S2, A2, B2, C2 = Node('s2'), Node('a2'), Node('b2'), Node('c2')
S2.add_neighbors(B2,A2)
A2.add_neighbors(B2,C2,G)
B2.add_children(A2)
C2.add_neighbors(G)

# edge cost/distance between nodes
edges = {
    (S,A):1, (S,B):4, (A,B):2, (A,C):5, (C,G):3, (A,G):12, (B,C):2,
    (S2,A2):1, (S2,B2):4, (A2,B2):2, (A2,C2):5, (C2,G):3, (A2,G):12, (B2,C2):2
}
def edge_cost(start, end):
    return edges.get((start, end), edges.get((end, start), None))

# heuristic cost/estimated distance to goal
heur = {S:7, A:6, B:4, C:2, G:0, S2:7, A2:6, B2:4, C2:2}
def heuristic(node, goal):
    return heur.get(node)

In [4]:
dijkstras(
    S, 
    neighbors_fnct=lambda node:set().union(node.neighbors).union(node.children), 
    goal=G, 
    distance_between_fnct=edge_cost, 
    #is_goal_reached_fnct=lambda p,g: p==g
)

{Node 's': [Node 's', Node 'a', Node 'b', Node 'c', Node 'g']}

In [5]:
greedy(
    S2, 
    neighbors_fnct=lambda n:set().union(n.neighbors).union(n.children), 
    goal=G, 
    heuristic_cost_estimate_fnct=heuristic
)

{Node 's2': [Node 's2', Node 'a2', Node 'g']}

In [6]:
astar(
    S, 
    neighbors_fnct=lambda n:set().union(n.neighbors).union(n.children), 
    goal=G, 
    distance_between_fnct=edge_cost, 
    heuristic_cost_estimate_fnct=heuristic
)

{Node 's': [Node 's', Node 'a', Node 'b', Node 'c', Node 'g']}

In [7]:
dijkstras(
    S2, 
    neighbors_fnct=lambda node:set().union(node.neighbors).union(node.children), 
    goal=G, 
    distance_between_fnct=edge_cost, 
    #is_goal_reached_fnct=lambda p,g: p==g
)

{Node 's2': [Node 's2', Node 'a2', Node 'c2', Node 'g']}

In [8]:
greedy(
    S, 
    neighbors_fnct=lambda n:set().union(n.neighbors).union(n.children), 
    goal=G, 
    heuristic_cost_estimate_fnct=heuristic
)

{Node 's': [Node 's', Node 'b', Node 'c', Node 'g']}

In [9]:
astar(
    [S, S2], 
    neighbors_fnct=lambda n:set().union(n.neighbors).union(n.children), 
    goal=G, 
    distance_between_fnct=edge_cost, 
    heuristic_cost_estimate_fnct=heuristic
)

{Node 's': [Node 's', Node 'a', Node 'b', Node 'c', Node 'g'],
 Node 's2': [Node 's2', Node 'a2', Node 'c2', Node 'g']}

In [10]:
dijkstras(
    Node._all.values(), 
    neighbors_fnct=lambda node:set().union(node.neighbors).union(node.children), 
    goal=G, 
    distance_between_fnct=edge_cost, 
)

{Node 's': [Node 's', Node 'a', Node 'b', Node 'c', Node 'g'],
 Node 'a': [Node 'a', Node 'b', Node 'c', Node 'g'],
 Node 'b': [Node 'b', Node 'c', Node 'g'],
 Node 'c': [Node 'c', Node 'g'],
 Node 'g': [Node 'g'],
 Node 's2': [Node 's2', Node 'a2', Node 'c2', Node 'g'],
 Node 'a2': [Node 'a2', Node 'c2', Node 'g'],
 Node 'c2': [Node 'c2', Node 'g'],
 Node 'b2': [Node 'b2', Node 'a2', Node 'c2', Node 'g']}

In [11]:
greedy(
    Node._all.values(), 
    neighbors_fnct=lambda n:set().union(n.neighbors).union(n.children), 
    goal=G, 
    heuristic_cost_estimate_fnct=heuristic
)

{Node 's': [Node 's', Node 'b', Node 'c', Node 'g'],
 Node 'b': [Node 'b', Node 'c', Node 'g'],
 Node 'c': [Node 'c', Node 'g'],
 Node 'g': [Node 'g'],
 Node 'a': [Node 'a', Node 'g'],
 Node 's2': [Node 's2', Node 'a2', Node 'g'],
 Node 'a2': [Node 'a2', Node 'g'],
 Node 'b2': [Node 'b2', Node 'a2', Node 'g'],
 Node 'c2': [Node 'c2', Node 'g']}

In [12]:
astar(
    Node._all.values(), 
    neighbors_fnct=lambda n:set().union(n.neighbors).union(n.children), 
    goal=G, 
    distance_between_fnct=edge_cost, 
    heuristic_cost_estimate_fnct=heuristic
)

{Node 's': [Node 's', Node 'a', Node 'b', Node 'c', Node 'g'],
 Node 'a': [Node 'a', Node 'b', Node 'c', Node 'g'],
 Node 'b': [Node 'b', Node 'c', Node 'g'],
 Node 'c': [Node 'c', Node 'g'],
 Node 'g': [Node 'g'],
 Node 's2': [Node 's2', Node 'a2', Node 'c2', Node 'g'],
 Node 'a2': [Node 'a2', Node 'c2', Node 'g'],
 Node 'c2': [Node 'c2', Node 'g'],
 Node 'b2': [Node 'b2', Node 'a2', Node 'c2', Node 'g']}

In [13]:
astar(
    #Node._all.values(), 
    [S,S2,B,C2,A],
    neighbors_fnct=lambda n:set().union(n.neighbors).union(n.children), 
    goal=G, 
    distance_between_fnct=edge_cost, 
    heuristic_cost_estimate_fnct=heuristic,
    reversePath=False
)

{Node 's': [Node 's', Node 'a', Node 'b', Node 'c', Node 'g'],
 Node 'a': [Node 'a', Node 'b', Node 'c', Node 'g'],
 Node 'b': [Node 'b', Node 'c', Node 'g'],
 Node 's2': [Node 's2', Node 'a2', Node 'c2', Node 'g'],
 Node 'c2': [Node 'c2', Node 'g']}

In [14]:
astar(
    #Node._all.values(), 
    [S,S2,B,C2,A],
    neighbors_fnct=lambda n:set().union(n.neighbors).union(n.children), 
    goal=G, 
    distance_between_fnct=edge_cost, 
    heuristic_cost_estimate_fnct=heuristic,
    reversePath=True
)

{Node 's': [Node 'g', Node 'c', Node 'b', Node 'a', Node 's'],
 Node 'a': [Node 'g', Node 'c', Node 'b', Node 'a'],
 Node 'b': [Node 'g', Node 'c', Node 'b'],
 Node 's2': [Node 'g', Node 'c2', Node 'a2', Node 's2'],
 Node 'c2': [Node 'g', Node 'c2']}